# Notebook 03 — Paper-Inspired Steering Ablations
## A toy state-space surrogate, not a reproduction of NA-RFM

### Purpose

This notebook turns the steering mechanism introduced in notebook `02` into a
controlled ablation lab. We study when a time-gated intervention changes an ODE
trajectory, how strongly it should be applied, and how to measure the resulting
control-versus-fidelity trade-off.

### Scope

The intervention in this notebook is added directly to the 2D velocity field:

Equation used in this notebook:
`d x / d t = v_theta(t, x) + λ * g(t) * (a - x)`


where `a` is a fixed control anchor, `g(t)` is a time gate, and `lambda` is the
steering strength.

This is **state-space steering**. It is not the internal activation intervention
used by arXiv:2602.11395. The paper learns a target-vs-rest RFM direction from
forward-process activations, adds that direction to a selected network block,
and combines it with a separate high-noise noise-alignment signal.

The toy field is retained because it makes timing, path deformation, collapse,
and metric design easy to inspect.

> In the steering paper, RFM means *Recursive Feature Machine*, not Flow
> Matching or Rectified Flow.

### Learning outcomes

After completing the notebook, you should be able to:

1. run a paired baseline-versus-steered comparison using identical initial
   samples;
2. distinguish intervention control from target-distribution fidelity;
3. compare early, middle, and late steering windows;
4. separate strength effects from schedule-shape effects;
5. identify collapse using diversity and target-reference metrics;
6. explain which components would be needed for genuine activation steering.

### Geometry used in the lab

- `circle`: 12 discrete support points;
- `disk`: a filled uniform disk;
- `swiss`: a thin, curved Swiss-roll distribution;
- `ring`: an annulus with a central void.

The origin is a **control anchor**, not a representation of these target
distributions. It is deliberately off-support for the circle and ring so that a
setting can improve control while damaging fidelity.

### Assessment evidence

Produce:

- one baseline plus a controlled timing, strength, and profile ablation;
- one table with control, target-fidelity, diversity, path, and runtime metrics;
- one fixed-seed comparison figure;
- one evidence-based conclusion and one stated limitation.

Quick defaults are suitable for checking the implementation. Graded conclusions
should use a baseline that visibly covers its target support.


In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torchdyn.core import NeuralODE


In [ ]:
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
def sample_noise(n):
    return torch.randn(n, 2, device=device)

def safe_float(x):
    return float(torch.as_tensor(x).detach().cpu().item())

def ensure_finite(name, tensor_like):
    arr = torch.as_tensor(tensor_like, dtype=torch.float32)
    if not torch.isfinite(arr).all():
        raise ValueError(f"Non-finite values detected in {name}")

def to_numpy(x):
    return x.detach().cpu().numpy()

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
})


In [ ]:
# Toy targets

def sample_circle_discrete(n, n_pts=12):
    angles = torch.linspace(0, 2 * math.pi, n_pts + 1, device=device)[:-1]
    xi = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
    idx = torch.randint(0, n_pts, (n,), device=device)
    return xi[idx]

def sample_disk_uniform(n):
    theta = 2 * math.pi * torch.rand(n, device=device)
    r = torch.sqrt(torch.rand(n, device=device))
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1)

def sample_swiss_roll(n, noise=0.3):
    t = 1.5 * math.pi * (1 + 2 * torch.rand(n, device=device))
    x = t * torch.cos(t) + noise * torch.randn(n, device=device)
    y = t * torch.sin(t) + noise * torch.randn(n, device=device)
    pts = torch.stack([x, y], dim=1)
    return pts / pts.abs().max() * 2

def sample_ring(n, center=(0.0, 0.0), radius=1.0, width=0.15):
    center = torch.as_tensor(center, device=device)
    theta = 2 * math.pi * torch.rand(n, device=device)
    r = radius + width * (torch.rand(n, device=device) - 0.5)
    return torch.stack([r * torch.cos(theta), r * torch.sin(theta)], dim=1) + center

TARGET_SPECS = {
    "circle": (sample_circle_discrete, torch.tensor([0.0, 0.0], device=device)),
    "disk": (sample_disk_uniform, torch.tensor([0.0, 0.0], device=device)),
    "swiss": (sample_swiss_roll, torch.tensor([0.0, 0.0], device=device)),
    "ring": (lambda n: sample_ring(n, radius=1.0, width=0.08), torch.tensor([0.0, 0.0], device=device)),
}

def get_target(name):
    return TARGET_SPECS[name]


In [ ]:
class VelocityMLP(nn.Module):
    def __init__(self, dim=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden), nn.SELU(),
            nn.Linear(hidden, hidden), nn.SELU(),
            nn.Linear(hidden, hidden), nn.SELU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, t, x, args=None):
        if t.dim() == 0:
            t = t.repeat(x.shape[0])
        return self.net(torch.cat([x, t[:, None]], dim=-1))


def ot_constant(x0, x1, t):
    t = t[:, None]
    return t * x1 + (1 - t) * x0, x1 - x0


class ODEWrapper(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, t, x, args=None):
        return self.net(t.repeat(x.shape[0]), x)


def train(sample_target_fn, n_steps=1500, batch_size=256, lr=1e-3, print_every=500):
    model = VelocityMLP().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    t0 = time.time()
    for step in range(1, n_steps + 1):
        x0 = sample_noise(batch_size)
        x1 = sample_target_fn(batch_size).to(device)
        t = torch.rand(batch_size, device=device)

        xt, ut = ot_constant(x0, x1, t)
        pred = model(t, xt)
        loss = ((pred - ut) ** 2).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        losses.append(safe_float(loss))
        if step % print_every == 0:
            elapsed = time.time() - t0
            t0 = time.time()
            print(f"step {step:>5d} | loss {loss.item():.4f} | {elapsed:.1f}s")

    return model, losses


In [ ]:
def integrate_model(
    model,
    n_samples=None,
    n_time=180,
    x0=None,
    wrap_fn=None,
    wrap_kwargs=None,
):
    """
    Integrate one model from a supplied initial-state bank.

    Passing x0 is required for paired baseline/steering comparisons. The
    n_samples fallback is retained for exploratory baseline plots.
    """
    if x0 is None:
        if n_samples is None:
            raise ValueError("Provide either x0 or n_samples")
        x0 = sample_noise(n_samples)

    if wrap_fn is None:
        ode_func = ODEWrapper(model)
    else:
        ode_func = wrap_fn(model=model, **(wrap_kwargs or {}))

    ode = NeuralODE(ode_func, solver="midpoint", sensitivity="adjoint")

    with torch.no_grad():
        traj = ode.trajectory(
            x0.clone(),
            t_span=torch.linspace(0, 1, n_time, device=x0.device),
        )

    return traj


def plot_loss(losses, title):
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(losses, alpha=0.3, lw=0.8)
    w = 200
    if len(losses) > w:
        ma = np.convolve(losses, np.ones(w) / w, mode="valid")
        ax.plot(range(w - 1, len(losses)), ma, lw=2, label=f"moving avg ({w})")
    ax.set_title(f"{title} | train loss")
    ax.set_xlabel("step")
    ax.set_ylabel("MSE")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()


In [ ]:
def steering_gate(t, start=0.2, end=0.8, profile="triangular"):
    t = t[:, None]
    active = ((t >= start) & (t <= end)).float()

    if profile == "constant":
        return active

    span = max(end - start, 1e-8)
    u = ((t - start) / span).clamp(0.0, 1.0)

    if profile == "triangular":
        w = torch.minimum(u, 1 - u).clamp(min=0.0) * 2
        return w * active
    if profile == "cosine":
        left = 0.5 - 0.5 * torch.cos(2 * math.pi * u)
        right = 0.5 - 0.5 * torch.cos(2 * math.pi * (1 - u))
        raw = torch.where(u <= 0.5, left, right)
        return raw * active

    raise ValueError("profile must be constant, triangular, or cosine")


def steering_gate_area(start, end, profile):
    """Analytic integral of the unit-height gate over ODE time."""
    span = float(end - start)
    if span <= 0:
        raise ValueError("end must be greater than start")
    if profile == "constant":
        return span
    if profile in {"triangular", "cosine"}:
        return 0.5 * span
    raise ValueError("profile must be constant, triangular, or cosine")


def plot_steering_profiles(start=0.70, end=0.95):
    t = torch.linspace(0, 1, 400, device=device)
    fig, ax = plt.subplots(figsize=(8, 3.5))

    for profile in ("constant", "triangular", "cosine"):
        gate = steering_gate(t, start=start, end=end, profile=profile)
        ax.plot(
            to_numpy(t),
            to_numpy(gate.squeeze(-1)),
            label=f"{profile}; area={steering_gate_area(start, end, profile):.3f}",
        )

    ax.axvspan(start, end, alpha=0.08)
    ax.set_xlabel("ODE time t")
    ax.set_ylabel("gate value")
    ax.set_title("Peak-normalized steering schedules")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.show()


def steered_velocity(model, x, t, anchor, steer_strength, start, end, profile):
    base = model(t, x)
    if steer_strength == 0.0:
        return base

    target = torch.as_tensor(anchor, device=x.device, dtype=x.dtype)
    if target.ndim == 1:
        target = target[None, :].expand_as(x)

    w = steering_gate(t, start=start, end=end, profile=profile)
    return base + steer_strength * w * (target - x)


class SteeredODEWrapper(nn.Module):
    def __init__(self, model, anchor, steer_strength, start, end, profile):
        super().__init__()
        self.model = model
        self.anchor = anchor
        self.steer_strength = float(steer_strength)
        self.start = float(start)
        self.end = float(end)
        self.profile = profile

    def forward(self, t, x, args=None):
        tt = t.repeat(x.shape[0]) if t.dim() == 0 else t
        return steered_velocity(
            self.model,
            x,
            tt,
            anchor=self.anchor,
            steer_strength=self.steer_strength,
            start=self.start,
            end=self.end,
            profile=self.profile,
        )


plot_steering_profiles()


In [ ]:
def trajectory_length(traj):
    step_displacements = torch.diff(traj, dim=0)
    step_lengths = torch.linalg.norm(step_displacements, dim=-1)
    return step_lengths.sum(dim=0)


def total_variance(samples):
    centered = samples - samples.mean(dim=0, keepdim=True)
    return torch.mean(torch.sum(centered**2, dim=-1))


def sliced_wasserstein_2d(samples, target_ref, n_projections=64):
    """
    Approximate 2-Wasserstein distance with evenly spaced 2D projections.
    Both sets are truncated to the same size before sorting.
    """
    n = min(samples.shape[0], target_ref.shape[0])
    x = samples[:n]
    y = target_ref[:n]

    angles = (
        torch.arange(n_projections, device=x.device, dtype=x.dtype)
        * math.pi
        / n_projections
    )
    directions = torch.stack(
        [torch.cos(angles), torch.sin(angles)],
        dim=1,
    )

    x_proj = torch.sort(x @ directions.T, dim=0).values
    y_proj = torch.sort(y @ directions.T, dim=0).values
    return torch.sqrt(torch.mean((x_proj - y_proj) ** 2))


def run_trajectory_experiment(traj, target_ref, target_anchor, name):
    """
    Report control and fidelity separately.

    Anchor metrics measure whether the intervention moved samples toward the
    control point. Target metrics measure whether the endpoint still resembles
    the target distribution.
    """
    end = traj[-1]
    target_ref = target_ref.to(device=end.device, dtype=end.dtype)
    anchor = torch.as_tensor(
        target_anchor,
        device=end.device,
        dtype=end.dtype,
    )

    ensure_finite(f"{name}: endpoint", end)
    ensure_finite(f"{name}: target reference", target_ref)

    anchor_dist = torch.linalg.norm(end - anchor, dim=-1)

    distance_matrix = torch.cdist(end, target_ref)
    generated_to_target = distance_matrix.min(dim=1).values.mean()
    target_to_generated = distance_matrix.min(dim=0).values.mean()
    symmetric_chamfer = generated_to_target + target_to_generated

    target_variance = total_variance(target_ref)
    generated_variance = total_variance(end)
    diversity_ratio = generated_variance / (target_variance + 1e-8)

    return {
        "name": name,
        "anchor_mean": safe_float(anchor_dist.mean()),
        "anchor_p90": safe_float(torch.quantile(anchor_dist, 0.90)),
        "generated_to_target": safe_float(generated_to_target),
        "target_to_generated": safe_float(target_to_generated),
        "target_chamfer": safe_float(symmetric_chamfer),
        "target_swd": safe_float(sliced_wasserstein_2d(end, target_ref)),
        "diversity_ratio": safe_float(diversity_ratio),
        "path_length": safe_float(trajectory_length(traj).mean()),
    }


def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()


def run_ablations(
    model,
    target_ref,
    target_anchor,
    x0_eval,
    configs,
    n_time=180,
):
    """
    Run an explicit, labeled configuration list with shared x0 and target data.
    """
    rows = []

    for cfg in configs:
        strength = float(cfg["strength"])
        start = float(cfg["start"])
        end = float(cfg["end"])
        profile = cfg["profile"]
        label = cfg["label"]

        synchronize_device()
        t0 = time.perf_counter()

        if strength == 0.0:
            traj = integrate_model(
                model,
                x0=x0_eval,
                n_time=n_time,
            )
        else:
            traj = integrate_model(
                model,
                x0=x0_eval,
                n_time=n_time,
                wrap_fn=SteeredODEWrapper,
                wrap_kwargs={
                    "anchor": target_anchor,
                    "steer_strength": strength,
                    "start": start,
                    "end": end,
                    "profile": profile,
                },
            )

        synchronize_device()
        wall_ms = (time.perf_counter() - t0) * 1000.0

        result = run_trajectory_experiment(
            traj,
            target_ref=target_ref,
            target_anchor=target_anchor,
            name=label,
        )
        result.update(
            {
                "label": label,
                "strength": strength,
                "start": start,
                "end": end,
                "profile": profile,
                "gate_area": (
                    0.0
                    if strength == 0.0
                    else steering_gate_area(start, end, profile)
                ),
                "nominal_dose": (
                    0.0
                    if strength == 0.0
                    else strength * steering_gate_area(start, end, profile)
                ),
                "wall_ms": wall_ms,
                "traj": to_numpy(traj),
            }
        )
        rows.append(result)

    return rows


In [ ]:
# Baseline over four targets
BASELINE_STEPS = 1200
BASELINE_BATCH = 256

experiment_state = {}

for name in ["circle", "disk", "swiss", "ring"]:
    sampler, anchor = get_target(name)
    print(f"\n==== Baseline on {name} ====")
    t_start = time.time()
    model, losses = train(sampler, n_steps=BASELINE_STEPS, batch_size=BASELINE_BATCH, print_every=400)
    target_ref = sampler(2048)
    traj = integrate_model(model, n_samples=2048, n_time=220)

    metrics = run_trajectory_experiment(
        traj,
        target_ref=target_ref,
        target_anchor=anchor,
        name=f"{name} baseline",
    )
    metrics["target"] = name
    metrics["train_s"] = time.time() - t_start
    metrics["loss_last"] = losses[-1]

    experiment_state[name] = {
        "sampler": sampler,
        "anchor": anchor,
        "target_ref": to_numpy(target_ref),
        "model": model,
        "traj": to_numpy(traj),
        "losses": losses,
        "metrics": metrics,
    }

    plot_loss(losses, name)
    print(metrics)


## Baseline target snapshots

In [ ]:
def plot_baseline_endpoints(experiment_state, n_show=900):
    fig, axes = plt.subplots(2, 2, figsize=(10, 10))
    axes = axes.ravel()

    for ax, (name, st) in zip(axes, experiment_state.items()):
        target_ref = np.asarray(st["target_ref"])[:n_show]
        generated = np.asarray(st["traj"])[-1, :n_show]

        ax.scatter(
            target_ref[:, 0],
            target_ref[:, 1],
            s=7,
            alpha=0.20,
            label="target reference",
        )
        ax.scatter(
            generated[:, 0],
            generated[:, 1],
            s=7,
            alpha=0.45,
            label="generated",
        )
        ax.scatter(
            st["anchor"][0].item(),
            st["anchor"][1].item(),
            s=80,
            marker="*",
            edgecolors="k",
            label="control anchor",
        )
        ax.set_title(name)
        ax.set_aspect('equal')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.legend(loc='best')
    plt.tight_layout()
    plt.show()

plot_baseline_endpoints(experiment_state)


## Steering ablations


## Steering schedules

The schedule controls *when* the state-space perturbation is active. Constant,
triangular, and cosine schedules have different integrated areas when their peaks
are equal:

- constant area: `end - start`;
- triangular area: `0.5 * (end - start)`;
- cosine area: `0.5 * (end - start)`.

A peak-matched profile comparison therefore changes both shape and total
intervention dose. The course configurations below match dose when comparing
profiles and report the nominal dose for every run.


In [ ]:
# t=0 is noise and t=1 is data.
EARLY_WINDOW = (0.05, 0.30)
MIDDLE_WINDOW = (0.375, 0.625)
LATE_WINDOW = (0.70, 0.95)

# For a window of width 0.25:
#   constant strength 0.2 -> dose 0.050
#   triangular strength 0.4 -> dose 0.050
#   cosine strength 0.4 -> dose 0.050
ABLATION_CONFIGS = [
    {
        "label": "baseline",
        "strength": 0.0,
        "start": 0.0,
        "end": 1.0,
        "profile": "constant",
    },
    {
        "label": "timing_early",
        "strength": 0.4,
        "start": EARLY_WINDOW[0],
        "end": EARLY_WINDOW[1],
        "profile": "triangular",
    },
    {
        "label": "timing_middle",
        "strength": 0.4,
        "start": MIDDLE_WINDOW[0],
        "end": MIDDLE_WINDOW[1],
        "profile": "triangular",
    },
    {
        "label": "timing_late",
        "strength": 0.4,
        "start": LATE_WINDOW[0],
        "end": LATE_WINDOW[1],
        "profile": "triangular",
    },
    {
        "label": "strength_low",
        "strength": 0.2,
        "start": LATE_WINDOW[0],
        "end": LATE_WINDOW[1],
        "profile": "triangular",
    },
    {
        "label": "strength_high",
        "strength": 0.8,
        "start": LATE_WINDOW[0],
        "end": LATE_WINDOW[1],
        "profile": "triangular",
    },
    {
        "label": "profile_constant_dose_matched",
        "strength": 0.2,
        "start": LATE_WINDOW[0],
        "end": LATE_WINDOW[1],
        "profile": "constant",
    },
    {
        "label": "profile_cosine_dose_matched",
        "strength": 0.4,
        "start": LATE_WINDOW[0],
        "end": LATE_WINDOW[1],
        "profile": "cosine",
    },
]

EVAL_SAMPLES = 1200
rows = []

for target_index, target in enumerate(["circle", "disk", "swiss", "ring"]):
    model = experiment_state[target]["model"]
    sampler, anchor = get_target(target)

    # Fixed evaluation banks for every configuration on this target.
    eval_seed = SEED + 10_000 + target_index
    torch.manual_seed(eval_seed)
    np.random.seed(eval_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(eval_seed)

    target_ref = sampler(EVAL_SAMPLES)
    x0_eval = sample_noise(EVAL_SAMPLES)

    target_rows = run_ablations(
        model,
        target_ref=target_ref,
        target_anchor=anchor,
        x0_eval=x0_eval,
        configs=ABLATION_CONFIGS,
        n_time=180,
    )

    for result in target_rows:
        result["target"] = target
        rows.append(result)

print(f"Collected {len(rows)} paired steering rows")


In [ ]:
def plot_trajectory_grid(rows, n_lines=20):
    # 4x2 layout: each target compares baseline and one non-zero steering setting by
    # label.
    by_target = {}
    for r in rows:
        by_target.setdefault(r["target"], []).append(r)

    fig, axes = plt.subplots(4, 2, figsize=(10, 16))
    for row, target in enumerate(["circle", "disk", "swiss", "ring"]):
        entries = by_target.get(target, [])
        base = next((r for r in entries if r["strength"] == 0.0), None)
        steer = next((r for r in entries if r["label"] == "timing_late"), None)

        for col, r in enumerate([base, steer]):
            ax = axes[row, col]
            if r is None:
                ax.set_axis_off()
                continue

            traj = np.asarray(r["traj"])
            idx = np.random.choice(traj.shape[1], size=min(n_lines, traj.shape[1]), replace=False)
            for j in idx:
                ax.plot(traj[:, j, 0], traj[:, j, 1], lw=0.7, alpha=0.45)
            ax.scatter(traj[0, :, 0], traj[0, :, 1], s=8, c="gray", alpha=0.25)
            ax.scatter(traj[-1, :, 0], traj[-1, :, 1], s=8, c="tab:blue", alpha=0.35)
            ax.set_aspect('equal')
            ax.set_xticks([])
            ax.set_yticks([])
            label = "baseline" if r["strength"] == 0.0 else r["label"]
            ax.set_title(f"{target} | {label}")
    plt.tight_layout()
    plt.show()

plot_trajectory_grid(rows, n_lines=18)


In [ ]:
QUALITY_TOLERANCE = 0.10

print(
    "Quality-constrained selection: lowest anchor distance among settings "
    "with <=10% SWD degradation from baseline"
)

for target in ["circle", "disk", "swiss", "ring"]:
    sub = [r for r in rows if r["target"] == target]
    baseline = next(r for r in sub if r["label"] == "baseline")

    print(f"\n{target}")
    print(
        f"  baseline | anchor={baseline['anchor_mean']:.4f} | "
        f"SWD={baseline['target_swd']:.4f} | "
        f"Chamfer={baseline['target_chamfer']:.4f} | "
        f"diversity={baseline['diversity_ratio']:.3f}"
    )

    for result in sub:
        result["swd_change_pct"] = (
            100.0
            * (result["target_swd"] - baseline["target_swd"])
            / max(baseline["target_swd"], 1e-8)
        )

    candidates = [
        r
        for r in sub
        if r["strength"] > 0
        and r["target_swd"] <= baseline["target_swd"] * (1.0 + QUALITY_TOLERANCE)
    ]

    if not candidates:
        print("  No steered setting passed the SWD quality constraint.")
    else:
        selected = min(candidates, key=lambda r: r["anchor_mean"])
        print(
            f"  selected: {selected['label']} | "
            f"anchor={selected['anchor_mean']:.4f} | "
            f"SWD={selected['target_swd']:.4f} "
            f"({selected['swd_change_pct']:+.1f}%) | "
            f"Chamfer={selected['target_chamfer']:.4f} | "
            f"diversity={selected['diversity_ratio']:.3f} | "
            f"path={selected['path_length']:.4f} | "
            f"dose={selected['nominal_dose']:.4f}"
        )

    print("  all configurations:")
    for result in sorted(sub, key=lambda r: r["target_swd"]):
        print(
            f"    {result['label']:<32} | "
            f"anchor={result['anchor_mean']:.4f} | "
            f"SWD={result['target_swd']:.4f} | "
            f"Chamfer={result['target_chamfer']:.4f} | "
            f"diversity={result['diversity_ratio']:.3f} | "
            f"path={result['path_length']:.4f} | "
            f"ms={result['wall_ms']:.0f}"
        )


## Analysis questions and submission checklist

### Interpretation questions

1. Which target shows the clearest case where anchor control improves while
   target fidelity deteriorates?
2. Does late steering preserve target SWD or diversity better than early
   steering at the same strength, duration, and profile?
3. Does increasing strength produce a monotonic improvement in control?
4. Which geometry is most sensitive to collapse, and what property of its
   support explains that sensitivity?
5. Do the dose-matched constant, triangular, and cosine schedules still differ?
6. Is the selected configuration genuinely better, or merely better under the
   chosen 10% SWD constraint?
7. Which result resembles the timing motivation in arXiv:2602.11395?
8. Why does that resemblance not make this notebook an implementation of
   NA-RFM?

### Required submission

Include:

- seed, device, training budget, sample count, and ODE resolution;
- the full paired results table;
- one fixed-seed baseline-versus-steered figure;
- one claim supported by at least two numerical measurements;
- one example of a control-versus-fidelity trade-off;
- one limitation;
- a short list of the missing NA-RFM components:
  - target-vs-rest activation collection;
  - learned RFM direction;
  - internal block intervention;
  - activation-norm scaling or amplification;
  - high-noise PCA noise alignment;
  - block and activation-collection-noise ablations.

For graded conclusions, repeat the baseline and selected configuration over at
least three seeds and report mean ± standard deviation.

See `COURSE_GUIDE.md` for the grader rubric and the advanced extension from this
state-space surrogate to genuine higher-dimensional activation steering.
